# Auditing MAVE coverage for RASopathy VUS: silent variant-matching failure modes

## What this is

An audit of whether multiplexed assay of variant effect (MAVE) data exists for the 40
RASopathy variants that are curated in ClinVar but remain VUS with no functional criterion
applied. A first pass, matching HGVS protein changes against MaveDB's `hgvs_pro` column,
reported zero coverage. That answer was wrong, and establishing why turned up five distinct
ways a plausible-looking cross-reference fails without raising an error.

**What this is not.** MaveDB already maps assay variants to human reference sequences and
retains pre- and post-mapped VRS representations; MaveMD resolves variants through the
ClinGen Allele Registry precisely so that transcript, identifier and coordinate-system
differences do not have to be handled by the user. None of that is news, and nothing here
claims otherwise. What this audit adds is a measurement: **how far a naive cross-reference
can be wrong on a real clinical panel when that mapped/VRS/CAID information is not used.**

## The five failure modes

**FM1 — Depositor numbering.** Score sets number positions against whatever construct was
assayed. HRAS score sets use the mature protein; RAF1 and BRAF depositions cover isolated
domains with offsets of +51 and +476. Matching against `hgvs_pro` returns a clean, confident
zero.

**FM2 — Family-level target names.** The target of `urn:mavedb:00000057` is named `RAS`, not
`HRAS`. `urn:mavedb:00000061` is named `RAF`, not `RAF1`. Target-name metadata therefore does
not provide a usable HGNC key for these score sets — not by equality, not by substring.

**FM3 — Full-text search misses real records.** Searching MaveDB for `HRAS` returns zero
score sets although four exist. This follows from FM2: the gene symbol is nowhere in the
searchable text. An inventory built from symbol search alone under-reports coverage.

**FM4 — Haplotypes.** Many records are multi-substitution haplotypes such as
`p.[Leu133Ala;Met170Leu]`. These fail in two directions at once: a VRS reader that ignores
`members` drops them entirely, while a substring match on `hgvs_pro` finds `Met170Leu` inside
them and attributes a combination score to a single variant.

**FM5 — Isoform mismatch.** KRAS `urn:mavedb:00000115` writes `p.Met170Leu` in `hgvs_pro`,
numbering against isoform 4B, while its own `postMapped` block maps to `NP_203524.1`, isoform
4A. On 4A the variant is not coding: the ClinGen Allele Registry renders it
`NP_203524.1:n.*62A>T`, sixty-two bases past the stop codon. The residue at position 170 in
the assayed protein is lysine, not methionine.

FM1–FM3 produce missed coverage. FM4 can produce either a missed measurement or an erroneous
single-variant attribution. FM5 produced the only **false-positive coverage call** in this
audit: seven score sets would have been recorded as measuring a variant that does not exist
in the protein they measured.

## The underlying question

The framing that survives all five is not about string matching:

> The question is not whether a variant string matches. It is whether the clinical variant
> exists on the protein that generated the score.

So the match here is keyed on `(protein accession, reference residue, position, alternate
residue)`, with the clinical side supplied by the ClinGen Allele Registry, which returns every
protein representation of a variant across isoforms — coding and non-coding alike. This is an
accession-aware audit of the same correspondence MaveMD resolves through CAIDs; it is not a
claim to a unique solution.

## Result

Three of forty variants have isolated MAVE measurements: HRAS I93V and G138S
(`urn:mavedb:00000057-a/b/c-1`) and BRAF I208V (`urn:mavedb:00000708-a-1`). Two more are
measured only inside haplotypes. The HGVS route found one variant, and that one was the KRAS
false positive.

Coverage is genuinely sparse. The point of the audit is not the count — it is that routes
which all look reasonable disagree with each other, and the disagreement is invisible without
an external anchor.

## Scope, and why the count is not the answer

This notebook establishes whether an assay exists and which protein it measured. It does not
judge whether the assay bears on pathogenicity — that depends on assay design and is read from
the source, not from identifiers.

Both recovered assays fail that second test. The HRAS score sets are a bacterial two-hybrid
system without membrane context using a single effector. The BRAF score set
(`urn:mavedb:00000708-a-1`) is a DHFR complementation screen measuring **domain stability in
yeast**, not RAS-MAPK signalling — and since the RASopathy mechanism is typically gain of
function, a normal stability score is not benign evidence and an abnormal one points the wrong
way. None of the five score sets carries a published calibration.

**So coverage is 3/40 and usable coverage is 0/40.** The audit's value is that the three are
correctly attributed, and the one the naive route returned was not.

*The BRAF match is verifiable by hand from the score set metadata: it declares
`UniProt: P15056 (offset 152)` over a 76-residue construct, so canonical residue 208 is
construct position 56, the declared sequence carries isoleucine there, and the record reads
`p.Ile56Val`. The original route searched `p.Ile208Val`.*

## 1. Setup

In [ ]:
import json, re, time, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 60)

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
OUT = WORK / "mave_coverage"; OUT.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "mave_cache"
for sub in ("scoresets", "mapped", "caid", "scores"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)

MAVEDB_API = "https://api.mavedb.org/api/v1"
CLINGEN_AR = "https://reg.clinicalgenome.org"
ATLAS_RAW = ("https://raw.githubusercontent.com/Fernandosr85/"
             "rasopathy-functional-evidence-atlas/main/outputs")

# Score sets whose targets are named at family level and are therefore invisible
# to any symbol-based search. Failure modes 2 and 3.
SEED_URNS = [f"urn:mavedb:00000057-{c}-1" for c in "abcd"] + \
            [f"urn:mavedb:00000061-{c}-1" for c in "abcdefghi"]

# Reference proteins seen in this panel. Resolution is by accession, never by
# gene symbol: KRAS NP_203524.1 is isoform 4A and diverges from clinical 4B
# numbering past residue ~150.
NP_GENE = {
    "NP_005334": "HRAS", "NP_789765": "HRAS",
    "NP_002871": "RAF1", "NP_001341618": "RAF1",
    "NP_004324": "BRAF", "NP_001341538": "BRAF",
    "NP_203524": "KRAS", "NP_004976": "KRAS", "NP_001356716": "KRAS",
    "NP_002515": "NRAS",
}

AA3to1 = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q",
          "Glu": "E", "Gly": "G", "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
          "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T", "Trp": "W",
          "Tyr": "Y", "Val": "V", "Sec": "U", "Pyl": "O", "Ter": "*"}
AA1to3 = {v: k for k, v in AA3to1.items()}

TRANSIENT = {429, 500, 502, 503, 504}


def cached(kind, key, fetch):
    # Transient upstream failures are never cached: a stored 504 poisons every
    # subsequent run of the notebook.
    fp = CACHE / kind / (re.sub(r"[^A-Za-z0-9._-]", "_", str(key)) + ".json")
    if fp.exists():
        return json.loads(fp.read_text())
    v = fetch()
    if not (isinstance(v, dict) and v.get("__error__") in TRANSIENT):
        try:
            fp.write_text(json.dumps(v))
        except OSError:
            pass
    return v


def locate(fn):
    hits = []
    for pat in (f"/kaggle/input/**/{fn}", f"/kaggle/working/**/{fn}", f"./**/{fn}"):
        hits += [Path(p) for p in glob.glob(pat, recursive=True)]
    hits = [p for p in hits if p.exists()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None


def from_atlas(fn):
    p = locate(fn)
    if p:
        return p
    dest = WORK / fn
    r = requests.get(f"{ATLAS_RAW}/{fn}", timeout=60)
    r.raise_for_status()
    dest.write_text(r.text)
    return dest


# Purge any transient errors left by earlier runs.
purged = 0
for sub in ("scoresets", "mapped", "caid"):
    for fp in (CACHE / sub).glob("*.json"):
        try:
            d = json.loads(fp.read_text())
        except Exception:
            fp.unlink(); purged += 1; continue
        if isinstance(d, dict) and d.get("__error__") in TRANSIENT:
            fp.unlink(); purged += 1
print(f"output: {OUT} | purged {purged} cached transient errors")

## 2. The 40 blocked variants

From the Atlas worksheet. These are ClinVar entries in the RASopathy panel that carry a
curation record but remain VUS with no functional criterion applied. The empty curation
columns (`binding_constraint`, `assay_reference`, `calibratable`, `notes`) pass through
untouched — they are filled by hand.

In [ ]:
blocked = pd.read_csv(from_atlas("blocked_vus_worksheet.csv"))
work = blocked.copy()
work["cid"] = work["VariationID"].astype(str).str.replace(r"\D", "", regex=True)
work["gene"] = work["GeneSymbol"].astype(str).str.strip().str.upper()


def parse_short(p):
    # "A91V" -> ("A", 91, "V"); three-letter forms accepted too.
    s = str(p).strip().replace("p.", "")
    m = re.fullmatch(r"([A-Z])(\d+)([A-Z*])", s)
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    m = re.fullmatch(r"([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|Ter)", s)
    if m:
        return AA3to1.get(m.group(1)), int(m.group(2)), AA3to1.get(m.group(3))
    return None, None, None


work[["ref_aa", "aa_pos", "alt_aa"]] = work["hgvs_p_short"].apply(
    lambda p: pd.Series(parse_short(p), index=["ref_aa", "aa_pos", "alt_aa"]))


def transcript_hgvs(name):
    # ClinVar Name: NM_002524.5(NRAS):c.272C>T (p.Ala91Val) -> NM_002524.5:c.272C>T
    m = re.match(r"^\s*([A-Z_0-9.]+)\s*\([^)]*\)\s*:\s*(c\.[^\s(]+)", str(name))
    if not m:
        m = re.match(r"^\s*([A-Z_0-9.]+)\s*:\s*(c\.[^\s(]+)", str(name))
    return f"{m.group(1)}:{m.group(2)}" if m else None


work["c_hgvs"] = work["Name"].apply(transcript_hgvs)

print(f"{len(work)} variants")
print(work["gene"].value_counts().to_string())
print(f"\ntranscript HGVS parsed : {work['c_hgvs'].notna().sum()}/{len(work)}")
print(f"residue triple parsed  : {work['aa_pos'].notna().sum()}/{len(work)}")
GENES_IN_SCOPE = sorted(work["gene"].unique())

## 3. Clinical side: every protein representation, from the ClinGen Allele Registry

The registry returns the canonical allele plus its representation on **every** transcript and
protein. That list is what makes failure mode 5 detectable: a variant can be `p.Met170Leu` on
one isoform and non-coding on another, and both appear side by side.

In [ ]:
def fetch_allele(hgvs, timeout=30):
    def go():
        r = requests.get(f"{CLINGEN_AR}/allele", params={"hgvs": hgvs}, timeout=timeout)
        if r.status_code != 200:
            return {"__error__": r.status_code, "__body__": r.text[:300]}
        return r.json()
    return cached("caid", hgvs, go)


def extract_caid(rec):
    if not isinstance(rec, dict) or rec.get("__error__"):
        return None
    m = re.search(r"(CA\d+)", rec.get("@id") or "")
    return m.group(1) if m else None


def extract_protein_reps(rec):
    # Every NP_ / ENSP representation, coding or not. Non-coding forms matter:
    # they are the signature of an isoform on which the variant does not exist.
    out = set()
    if not isinstance(rec, dict):
        return out
    for ta in (rec.get("transcriptAlleles") or []):
        pe = (ta or {}).get("proteinEffect") or {}
        h = pe.get("hgvs")
        if isinstance(h, str):
            out.add(h)
    return out


caids, reps = [], []
for i, r in enumerate(work.itertuples(), 1):
    rec = fetch_allele(r.c_hgvs) if r.c_hgvs else {}
    caids.append(extract_caid(rec))
    reps.append(";".join(sorted(extract_protein_reps(rec))) or None)
    if i % 10 == 0:
        print(f"  {i}/{len(work)}", flush=True)
    time.sleep(0.15)
work["caid"] = caids
work["protein_reps"] = reps
print(f"\nCAID resolved            : {work['caid'].notna().sum()}/{len(work)}")
print(f"protein representations  : {work['protein_reps'].notna().sum()}/{len(work)}")
work.to_csv(OUT / "1_variants_resolved.csv", index=False)

In [ ]:
def np_keys(protein_reps):
    # Coding protein keys only: (NP accession, ref, pos, alt).
    out = set()
    for h in str(protein_reps or "").split(";"):
        m = re.match(r"(NP_[\d.]+):p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|Ter)", h.strip())
        if m:
            out.add((m.group(1), AA3to1.get(m.group(2)), int(m.group(3)),
                     AA3to1.get(m.group(4))))
    return out


def noncoding_nps(protein_reps):
    # Isoforms where the same variant is not coding, e.g. NP_203524.1:n.*62A>T.
    return {m.group(1) for h in str(protein_reps or "").split(";")
            if (m := re.match(r"(NP_[\d.]+):n\.", h.strip()))}


work["n_coding_isoforms"] = work["protein_reps"].apply(lambda p: len(np_keys(p)))
work["noncoding_on"] = work["protein_reps"].apply(
    lambda p: ";".join(sorted(noncoding_nps(p))) or None)

mixed = work[work["noncoding_on"].notna() & (work["n_coding_isoforms"] > 0)]
print(f"variants coding on some isoforms and non-coding on others: {len(mixed)}")
if len(mixed):
    print("These are where a gene-level match can attribute the wrong score:\n")
    print(mixed[["gene", "cid", "hgvs_p_short", "n_coding_isoforms",
                 "noncoding_on"]].to_string(index=False))

## 4. Assay side: building the score-set inventory

Two sources, because neither is sufficient. Full-text search finds most score sets but
returns nothing for genes whose targets are named at family level; the seed list supplies
those. Target names are matched by token rather than equality, since they read
`RAF1 RAF-like Ras-binding domain`.

Note that the gene assigned here is provisional. The authoritative assignment comes later,
from the reference protein in the mapped variants.

In [ ]:
def as_list(resp):
    if isinstance(resp, list):
        return resp
    if isinstance(resp, dict):
        for key in ("scoreSets", "score_sets", "results", "items", "data"):
            v = resp.get(key)
            if isinstance(v, list):
                return v
    return []


def search_score_sets(text, limit=100):
    # The endpoint 422s above its own cap, which has changed between versions.
    def go():
        for body in ({"text": text, "limit": limit}, {"text": text, "limit": 50},
                     {"text": text}):
            r = requests.post(f"{MAVEDB_API}/score-sets/search", json=body, timeout=60)
            if r.status_code == 200:
                return r.json()
        print(f"  ! search failed for {text!r}")
        return {"scoreSets": []}
    return as_list(cached("scoresets", f"search_{text}", go))


def score_set(urn):
    def go():
        r = requests.get(f"{MAVEDB_API}/score-sets/{urn}", timeout=60)
        return {"__error__": r.status_code} if r.status_code != 200 else r.json()
    return cached("scoresets", f"detail_{urn}", go)


def target_names(ss):
    out = set()
    for tg in (ss.get("targetGenes") or []):
        if not isinstance(tg, dict):
            continue
        for key in ("name", "geneName"):
            v = tg.get(key)
            if isinstance(v, str):
                out.add(v.strip().upper())
    return out


def gene_in_targets(gene, targets):
    return any(gene in re.split(r"[^A-Z0-9]+", t.upper()) for t in targets)


rows = []
for g in GENES_IN_SCOPE:
    hits = search_score_sets(g)
    n_ok = 0
    for ss in hits:
        if not isinstance(ss, dict):
            continue
        tn = target_names(ss)
        ok = gene_in_targets(g, tn)
        n_ok += int(ok)
        rows.append({"provisional_gene": g, "urn": ss.get("urn"), "title": ss.get("title"),
                     "target_names": ";".join(sorted(tn)) or None, "target_match": ok,
                     "num_variants": ss.get("numVariants"), "source": "search"})
    print(f"  {g}: {len(hits)} search hits, {n_ok} with matching target name")

seen = {r["urn"] for r in rows}
for urn in SEED_URNS:
    if urn in seen:
        continue
    ss = score_set(urn)
    if ss.get("__error__"):
        print(f"  {urn}: error {ss['__error__']}"); continue
    tn = target_names(ss)
    rows.append({"provisional_gene": None, "urn": urn, "title": ss.get("title"),
                 "target_names": ";".join(sorted(tn)) or None, "target_match": True,
                 "num_variants": ss.get("numVariants"), "source": "seed"})
    print(f"  seeded {urn}: target named {tn} — no gene symbol present")

inv = pd.DataFrame(rows).drop_duplicates("urn")
inv.to_csv(OUT / "2_scoreset_inventory.csv", index=False)

confirmed = inv[inv["target_match"]].copy()
urns = confirmed["urn"].dropna().tolist()
print(f"\nscore sets to download: {len(urns)} "
      f"({(confirmed['source'] == 'seed').sum()} invisible to search)")

discarded = inv[~inv["target_match"]]
print(f"discarded on target mismatch: {len(discarded)}")
if len(discarded):
    print(discarded.groupby("provisional_gene")["target_names"].apply(
        lambda s: ";".join(sorted({x for v in s.dropna() for x in v.split(";")}))[:90]
    ).to_string())

## 5. Mapped variants: alleles, haplotypes, and the reference protein

`postMapped` is reference-relative, which fixes failure mode 1. It is either an `Allele` or a
`Haplotype` holding several members; reading `location` and `state` from the top level drops
every haplotype, which is failure mode 4.

The `sequenceReference.label` names the protein the assay actually measured, which is
what exposes failure mode 5. Target-name metadata cannot serve that role (FM2).

In [ ]:
def mapped_variants(urn):
    def go():
        r = requests.get(f"{MAVEDB_API}/score-sets/{urn}/mapped-variants", timeout=300)
        return {"__error__": r.status_code} if r.status_code != 200 else r.json()
    return cached("mapped", urn, go)


def substitutions(rec):
    # Yield one row per single-residue substitution. Haplotype members are
    # emitted individually but flagged: their score measures the combination,
    # not the substitution.
    pm = rec.get("postMapped") or {}
    members = pm.get("members") or []
    blocks = members if members else [pm]
    out = []
    for b in blocks:
        loc = b.get("location") or {}
        pos = loc.get("end")
        alt = ((b.get("state") or {}).get("sequence") or "")
        np_acc = (loc.get("sequenceReference") or {}).get("label")
        ref = next((e.get("value") for e in (b.get("extensions") or [])
                    if e.get("name") == "vrs_ref_allele_seq"
                    and isinstance(e.get("value"), str)), None)
        if pos is None or len(alt) != 1 or not ref or len(ref) != 1:
            continue
        out.append({"ref_aa": ref.upper(), "aa_pos": int(pos), "alt_aa": alt.upper(),
                    "np_acc": np_acc,
                    "context": "haplotype" if members else "allele",
                    "n_members": len(members) if members else 1})
    return out


def gene_of_np(acc):
    return NP_GENE.get(str(acc).split(".")[0])


# Retry transient failures inline: a 504 that slips through leaves a score set
# out of the totals while looking, in the output, like a missing mapping.
MAX_RETRY = 3

rows, failures, unknown_np = [], [], set()
for n, urn in enumerate(urns, 1):
    mv, attempt = None, 0
    while attempt < MAX_RETRY:
        mv = mapped_variants(urn)
        if not (isinstance(mv, dict) and mv.get("__error__") in TRANSIENT):
            break
        attempt += 1
        print(f"     retry {attempt}/{MAX_RETRY} after HTTP {mv['__error__']}", flush=True)
        time.sleep(5 * attempt)
    if isinstance(mv, dict) and mv.get("__error__"):
        failures.append({"urn": urn, "error": mv["__error__"],
                         "kind": "transient" if mv["__error__"] in TRANSIENT
                                 else "no mapping"})
        print(f"  {n}/{len(urns)} {urn}: FAILED HTTP {mv['__error__']}")
        continue
    recs = mv if isinstance(mv, list) else (mv or {}).get("mappedVariants", [])
    kept = 0
    for rec in recs:
        for s in substitutions(rec):
            g = gene_of_np(s["np_acc"])
            if g is None:
                if s["np_acc"]:
                    unknown_np.add(s["np_acc"])
                continue
            rows.append({"urn": urn, "gene": g, **s})
            kept += 1
    print(f"  {n}/{len(urns)} {urn}: {len(recs)} records -> {kept} substitutions")

mapped = pd.DataFrame(rows).drop_duplicates()
mapped.to_csv(OUT / "3_mapped_substitutions.csv", index=False)
print(f"\ntotal substitutions: {len(mapped):,}")
print(mapped["context"].value_counts().to_string())
print("\nreference protein per score set:")
print(mapped.groupby(["urn", "gene"])["np_acc"]
      .agg(lambda s: ";".join(sorted(set(s)))).to_string())

TRANSIENT_FAILURES = []
if failures:
    fdf = pd.DataFrame(failures)
    perm = fdf[fdf["kind"] == "no mapping"]
    trans = fdf[fdf["kind"] == "transient"]
    TRANSIENT_FAILURES = trans["urn"].tolist()
    if len(perm):
        print("\nno published mapping (404) - the protein-anchored route is unavailable "
              "for these, and no coverage claim can be made either way:")
        print(perm[["urn", "error"]].to_string(index=False))
    if len(trans):
        print("\n*** TRANSIENT FAILURES after retries. These score sets are MISSING from "
              "the totals above; rerun this cell before freezing any number: ***")
        print(trans[["urn", "error"]].to_string(index=False))
if unknown_np:
    print(f"\nproteins missing from NP_GENE (add them): {sorted(unknown_np)}")

## 6. The match

Keyed on `(protein accession, reference residue, position, alternate residue)`. Both sides
of the key are anchored to a RefSeq protein, so isoform divergence cannot produce a spurious
hit.

Isolated measurements and haplotype-only measurements are reported separately: a
substitution seen only inside a haplotype has no isolated score.

In [ ]:
key = ["np_acc", "ref_aa", "aa_pos", "alt_aa"]
mapped["aa_pos"] = mapped["aa_pos"].astype(int)

solo = mapped[mapped["context"] == "allele"]
haplo = mapped[mapped["context"] == "haplotype"]
map_solo = solo.groupby(key)["urn"].apply(lambda s: ";".join(sorted(set(s)))).to_dict()
map_haplo = haplo.groupby(key)["urn"].apply(lambda s: ";".join(sorted(set(s)))).to_dict()

iso_hits, sets_solo, sets_haplo = [], [], []
for reps in work["protein_reps"]:
    ks = np_keys(reps)
    s_urns = {u for k in ks if (v := map_solo.get(k)) for u in v.split(";")}
    h_urns = {u for k in ks if (v := map_haplo.get(k)) for u in v.split(";")}
    matched_np = sorted({k[0] for k in ks if k in map_solo or k in map_haplo})
    iso_hits.append(";".join(matched_np) or None)
    sets_solo.append(";".join(sorted(s_urns)) or None)
    sets_haplo.append(";".join(sorted(h_urns - s_urns)) or None)

work["matched_protein"] = iso_hits
work["score_sets_isolated"] = sets_solo
work["score_sets_haplotype_only"] = sets_haplo
work["assay_available"] = work["score_sets_isolated"].notna()

show = ["gene", "cid", "hgvs_p_short", "caid", "matched_protein",
        "score_sets_isolated", "score_sets_haplotype_only"]
print(f"isolated measurements   : {int(work['assay_available'].sum())}/{len(work)}")
print(f"haplotype-only          : {int(work['score_sets_haplotype_only'].notna().sum())}")
print()
print(work[work["assay_available"] | work["score_sets_haplotype_only"].notna()][show]
      .to_string(index=False))

In [ ]:
# What the original HGVS-protein route would have returned, and whether each of
# its hits is real. This is where failure mode 5 shows up as a false positive.
def scores_csv(urn):
    fp = CACHE / "scores" / f"{re.sub(r'[^A-Za-z0-9._-]', '_', urn)}.csv"
    if not fp.exists():
        r = requests.get(f"{MAVEDB_API}/score-sets/{urn}/scores", timeout=300)
        if r.status_code != 200:
            return pd.DataFrame()
        fp.write_text(r.text)
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.DataFrame()


urn_gene = mapped.groupby("urn")["gene"].first().to_dict()
naive = {}
for urn in urns:
    sc = scores_csv(urn)
    if "hgvs_pro" not in sc.columns:
        continue
    pros = set(sc["hgvs_pro"].dropna().astype(str))
    g = urn_gene.get(urn)
    for r in work[work["gene"] == g].itertuples():
        rf, ps, al = r.ref_aa, r.aa_pos, r.alt_aa
        if not rf or pd.isna(ps):
            continue
        three = f"p.{AA1to3.get(rf, rf)}{int(ps)}{AA1to3.get(al, al)}"
        if any(three in p or str(r.hgvs_p_short) in p for p in pros):
            naive.setdefault(r.cid, set()).add(urn)

work["hgvs_route_sets"] = work["cid"].map(
    lambda c: ";".join(sorted(naive.get(c, []))) or None)
work["hgvs_route_hit"] = work["hgvs_route_sets"].notna()

n_new = int(work["assay_available"].sum())
n_old = int(work["hgvs_route_hit"].sum())
false_pos = work[work["hgvs_route_hit"] & ~work["assay_available"]]
missed = work[work["assay_available"] & ~work["hgvs_route_hit"]]

print(f"protein-anchored route : {n_new}/{len(work)}")
print(f"HGVS route (original)  : {n_old}/{len(work)}")
print(f"\nmissed by the HGVS route: {len(missed)}")
if len(missed):
    print(missed[show].to_string(index=False))
print(f"\nFALSE POSITIVES from the HGVS route: {len(false_pos)}")
if len(false_pos):
    print(false_pos[["gene", "cid", "hgvs_p_short", "hgvs_route_sets",
                     "noncoding_on"]].to_string(index=False))
    print("\nThese matched on depositor numbering only. Check `noncoding_on`: if the "
          "assayed protein is listed there, the variant does not exist in it.")
work.to_csv(OUT / "4_route_comparison.csv", index=False)

### 6b. The four routes, side by side

The claim that plausible routes disagree should be measured, not asserted. Each row below is a
cross-reference someone might reasonably build; all four run against the same 40 variants.

- **A — symbol search only.** Inventory from full-text search, no seeded URNs. Fails on FM2/FM3.
- **B — `hgvs_pro` substring.** The original route. Fails on FM1, FM4, FM5.
- **C — post-mapped, top-level only.** Reads `postMapped.location`/`state` without descending
  into haplotype `members`. Fails on FM4.
- **D — accession-aware.** The route used here.

In [ ]:
# Route A: what the inventory would have covered without the seeded URNs.
search_only = set(inv.loc[(inv["source"] == "search") & inv["target_match"], "urn"])
map_A = (mapped[(mapped["context"] == "allele") & mapped["urn"].isin(search_only)]
         .groupby(key)["urn"].apply(lambda s: ";".join(sorted(set(s)))).to_dict())

# Route C: post-mapped read that never descends into haplotype members. Identical
# to D for single alleles; differs wherever a variant is only in a haplotype.
map_C = map_solo


def covered_set(m):
    # Compare routes as sets of variants, not as counts. Two routes can each
    # return one variant and disagree completely about which one.
    return frozenset(r.cid for r in work.itertuples()
                     if any(k in m for k in np_keys(r.protein_reps)))


sets_by_route = {
    "A. symbol search only":          covered_set(map_A),
    "B. hgvs_pro substring":          frozenset(work.loc[work["hgvs_route_hit"], "cid"]),
    "C. post-mapped, top level only": covered_set(map_C),
    "D. accession-aware":             frozenset(work.loc[work["assay_available"], "cid"]),
}
FAILS_ON = {"A. symbol search only": "FM2, FM3",
            "B. hgvs_pro substring": "FM1, FM4, FM5",
            "C. post-mapped, top level only": "FM4",
            "D. accession-aware": "-"}
truth = sets_by_route["D. accession-aware"]
label = dict(zip(work["cid"], work["hgvs_p_short"]))

routes = pd.DataFrame([
    {"route": r, "covered": len(s), "fails_on": FAILS_ON[r],
     "false_positives": len(s - truth), "missed_vs_D": len(truth - s),
     "variants": ";".join(sorted(label.get(c, c) for c in s)) or "-"}
    for r, s in sets_by_route.items()])
routes.to_csv(OUT / "7_route_table.csv", index=False)
print(routes.to_string(index=False))

distinct = len(set(sets_by_route.values()))
agree = [r for r, s in sets_by_route.items() if s == truth]
print(f"\ndistinct answer sets across the four routes: {distinct}")
print(f"routes agreeing with D: {agree}")
disjoint = [(a, b) for a in sets_by_route for b in sets_by_route
            if a < b and sets_by_route[a] and sets_by_route[b]
            and not (sets_by_route[a] & sets_by_route[b])]
for a, b in disjoint:
    print(f"  {a} and {b} share no variant at all "
          f"({len(sets_by_route[a])} vs {len(sets_by_route[b])} covered)")

## 7. Calibration state

Three states, kept distinct. A calibration flagged research-use-only signals exploratory
method or limited clinical control data and does not by itself support clinical
classification; collapsing it into "calibrated" would be misleading in a table going to a
VCEP.

In [ ]:
relevant = sorted({u for col in ("score_sets_isolated", "score_sets_haplotype_only")
                   for s in work[col].dropna() for u in s.split(";")})
print(f"score sets covering at least one of the 40: {len(relevant)}")

rows = []
for urn in relevant:
    ss = score_set(urn)
    if ss.get("__error__"):
        rows.append({"urn": urn, "state": "unknown", "note": f"HTTP {ss['__error__']}"})
        continue
    cal = ss.get("scoreCalibrations") or ss.get("calibrations") or []
    if isinstance(cal, dict):
        cal = [cal]
    if not cal:
        rows.append({"urn": urn, "title": ss.get("title"), "state": "uncalibrated"})
        continue
    for c in cal:
        blob = json.dumps(c)
        rows.append({"urn": urn, "title": ss.get("title"),
                     "state": "RUO" if c.get("researchUseOnly") else "calibrated",
                     "calibration_title": c.get("title"),
                     "is_primary": c.get("isPrimary") or c.get("primary"),
                     "evidence_strengths": ";".join(sorted(set(re.findall(
                         r"(Very Strong|Strong|Moderate|Supporting)", blob)))) or None,
                     "has_oddspath": bool(re.search(r"odds[_ ]?path", blob, re.I))})
cal_tab = pd.DataFrame(rows)
cal_tab.to_csv(OUT / "5_calibration_state.csv", index=False)
print(cal_tab.to_string(index=False) if len(cal_tab) else "none")

## 8. Table for the VCEP

In [ ]:
def calibration_of(sets):
    if pd.isna(sets) or not str(sets).strip():
        return "no assay"
    if not len(cal_tab) or "urn" not in cal_tab.columns:
        return "unknown"
    states = set()
    for u in str(sets).split(";"):
        sub = cal_tab[cal_tab["urn"] == u]
        states.update(sub["state"].tolist() if len(sub) else ["unknown"])
    for pref in ("calibrated", "RUO", "uncalibrated", "unknown"):
        if pref in states:
            return pref
    return "unknown"


final = work.copy()
final["calibration_status"] = final["score_sets_isolated"].map(calibration_of)
final["measurement_context"] = np.where(
    final["assay_available"], "isolated",
    np.where(final["score_sets_haplotype_only"].notna(),
             "in combination only", "none"))

out_cols = [c for c in [
    "GeneSymbol", "VariationID", "hgvs_p_short", "Name", "caid", "clinvar_url",
    "binding_constraint", "assay_available", "measurement_context",
    "matched_protein", "score_sets_isolated", "score_sets_haplotype_only",
    "calibration_status", "hgvs_route_hit", "noncoding_on",
    "assay_reference", "calibratable", "notes"] if c in final.columns]
final[out_cols].to_csv(OUT / "6_vcep_table.csv", index=False)
print(final[out_cols].to_string(index=False))

In [ ]:
manifest = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_variants": int(len(work)),
    "coverage": {
        "protein_anchored_isolated": int(work["assay_available"].sum()),
        "haplotype_only": int(work["score_sets_haplotype_only"].notna().sum()),
        "hgvs_route": int(work["hgvs_route_hit"].sum()),
        "hgvs_route_false_positives": int(
            (work["hgvs_route_hit"] & ~work["assay_available"]).sum()),
        "missed_by_hgvs_route": int(
            (work["assay_available"] & ~work["hgvs_route_hit"]).sum()),
    },
    "inventory": {
        "score_sets_downloaded": len(urns),
        "invisible_to_search": int((confirmed["source"] == "seed").sum()),
        "substitutions_total": int(len(mapped)),
        "substitutions_in_haplotypes": int((mapped["context"] == "haplotype").sum()),
    },
    "reference_proteins": sorted(set(mapped["np_acc"].dropna())),
    "routes": {r: {"covered": len(v), "variants": sorted(label.get(c, c) for c in v)}
               for r, v in sets_by_route.items()},
    "distinct_answer_sets": int(len(set(sets_by_route.values()))),
    "transient_failures_excluded_from_totals": TRANSIENT_FAILURES,
    "calibration_states": (cal_tab["state"].value_counts().to_dict()
                           if len(cal_tab) else {}),
    "failure_modes": [
        "1. Depositor numbering differs from reference; postMapped resolves it. (missed coverage)",
        "2. Target-name metadata is family-level (RAS, RAF); it provides no usable HGNC key. (missed coverage)",
        "3. Full-text search returns zero for genes that have score sets. (missed coverage)",
        "4. Haplotypes: a VRS reader ignoring members drops them; a substring match "
        "on hgvs_pro misattributes combination scores to single variants.",
        "5. Isoform mismatch: hgvs_pro and postMapped can reference different "
        "isoforms of the same gene. Sole source of a false-positive coverage "
        "call in this audit.",
    ],
    "caveats": [
        ("TOTALS INCOMPLETE: {} score set(s) failed transiently and are missing from "
         "the substitution counts. Rerun before citing them.").format(
             len(TRANSIENT_FAILURES)) if TRANSIENT_FAILURES else
        "All inventoried score sets with a published mapping were retrieved.",
        "Existence of an assay is not fitness for germline classification.",
        "RUO calibration is not equivalent to calibrated.",
        "Nothing here constitutes a variant classification. No score set involved "
        "carries a published calibration, so no failure mode here altered a "
        "classification - only coverage ascertainment.",
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("\nfiles:")
for f in sorted(OUT.iterdir()):
    print("  ", f.name)

## 9. Reading the output

**`coverage.hgvs_route_false_positives` is the number that matters most.** A false negative
costs an opportunity; a false positive attributes an experimental measurement to a variant
that was never measured. If it is non-zero, inspect `noncoding_on` for those rows — the
assayed protein appearing there is the signature of failure mode 5.

**`inventory.invisible_to_search`** quantifies failure modes 2 and 3 together: score sets
that exist, cover panel genes, and cannot be found by searching for the gene.

**`measurement_context`** must survive into anything sent onward. "In combination only" is
neither "measured" nor "no assay", and merging it into either would misrepresent the record.

**What this notebook does not settle — and it is the larger half.** Whether a recovered assay
bears on pathogenicity depends on assay design, which is read from the source publication, not
from identifiers. Both recovered assays fail on design:

- HRAS `00000057-a/b/c-1`: bacterial two-hybrid, no membrane context, one effector, ITC
  validation explicitly noisier near wild-type binding — which is where both variants sit.
- BRAF `00000708-a-1`: DHFR complementation measuring domain stability in yeast. The
  RASopathy mechanism is typically gain of function; a stability readout detects
  destabilisation, so a normal score is not benign evidence and an abnormal score points the
  wrong way. The score set leaves `Assay type`, `Molecular mechanism` and `Model system`
  unspecified, so the mismatch is visible only from the abstract.

Anything sent onward needs a mechanism column beside `assay_available`. Reporting three
variants as having an assay, without it, invites precisely the wrong inference.